 ...continued from the PPT

# Loading Data

## Baisc Setup

First, let's import a few common modules, ensure MatplotLib plots figures inline and prepare a function to save the figures. We also check that Python 3.5 or later is installed, as well as Scikit-Learn ≥0.20.

In [ ]:
# Python ≥3.5 is required
import sys

# Scikit-Learn ≥0.20 is required
import sklearn

# Common imports
import numpy as np
import os

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

# Where to save the figures
PROJECT_ROOT_DIR = "."
IMAGES_PATH = os.path.join(PROJECT_ROOT_DIR, "images")
os.makedirs(IMAGES_PATH, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = os.path.join(IMAGES_PATH, fig_id + "." + fig_extension)
    print("Saving figure", fig_id)
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

## Get the data

In [ ]:
import os
import pandas as pd
# import tarfile
# import urllib.request

housing = pd.read_csv(os.path.join("dataset", "housing.csv"))
housing.head(10)

Each row represents one district and there are 10 attributes. The `info()` method is useful to get a quick description of the data, in particular the total number of rows, each attribute’s type, and the number of nonnull values.

In [ ]:
housing.info()

There are 20,640 instances in the dataset. It is fairly small by machine learning standards, but it’s perfect to get started. Notice that the `total_bedrooms` attribute has only 20,433 nonnull values, meaning that 207 districts are **missing** this feature. We will need to take care of this later.

All attributes are numerical, except the `ocean_proximity` field. Its type is 
**object**, so it could hold any kind of Python object, in particular, text. When we looked at the top five rows, you probably noticed that the values in the `ocean_proximity` column were repetitive, which means that it is probably a **categorical attribute**. We can find out what categories exist and how many districts belong to each category by using the `value_counts()` method:

In [ ]:
housing["ocean_proximity"].value_counts()

The `describe()` method shows a summary of the numerical attributes.

In [ ]:
housing.describe()

Another quick way to get a feel of the type of data you are dealing with is to plot a
**histogram** for each numerical attribute. A histogram shows the number of instances (on the vertical axis) that have a given value range (on the horizontal axis).

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
housing.hist(bins=10, figsize=(20,15))
save_fig("attribute_histogram_plots")
plt.show()

There are two things we might notice.

1. **The median income attribute does not look like it is expressed in US dollars (USD).** After checking with the team that collected the data, we are told that the data has been scaled and capped at 15 for higher median incomes, and at 0.5 for lower median incomes. The numbers represent roughly tens of thousands of dollars (e.g., 3 actually means about $30,000).

2. **The housing median age and the median house value were also capped.** The machine learning algorithms may learn that prices never go beyond that limit. We need to check with the client team if this is a problem or not. If they tell us that they need precise predictions even beyond $500,000, then we have two options:

+ Collect proper labels for the districts whose labels were capped.
+ Remove those districts from the training set (and also from the test set).

## Create a Test Set

Creating a test set is theoretically simple: pick some instances randomly, typically 20% of the dataset (or less if your dataset is very large), and set them aside. We can use the `train_test_split()` function from Scikit-Learn.

In [ ]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(housing, test_size=0.2, random_state=100)

In [ ]:
test_set.head()

In [ ]:
len(train_set)

In [ ]:
len(test_set)

# Data Exploration

## Scatterplot

First, make sure we have put the test set aside and we are **only exploring the training set**. Also, if the training set is very large, we may want to sample an exploration set, to make manipulations easy and fast. In our case, the set is quite small, so we can just work directly on the full set. Let’s create a copy so that we can play with it without harming the training set:

In [ ]:
housing = train_set.copy()

Since there is geographical information (latitude and longitude), it is a good idea to create a **scatterplot** of all districts to visualize the data.

In [ ]:
housing.plot(kind="scatter", x="longitude", y="latitude")
save_fig("bad_visualization_plot")

Setting the `alpha` option to 0.1 makes it much easier to visualize the places where there is a high density of data points.

In [ ]:
housing.plot(kind="scatter", x="longitude", y="latitude", alpha=0.1)
save_fig("better_visualization_plot")

Now include the information of the housing prices.

In [ ]:
housing.plot(kind="scatter", x="longitude", y="latitude", alpha=0.4,
             s=housing["population"]/100, label="population", figsize=(10,7),
             c="median_house_value", # color depends on median_house_value
             cmap=plt.get_cmap("jet"), colorbar=True,
             sharex=False)
plt.legend()
save_fig("housing_prices_scatterplot")

- The radius of each circle represents the district’s population (option `s`).
- The color represents the price (option `c`).
- We will use a predefined color map (option `cmap`) called `jet`, which ranges from blue (low values) to red (high prices).

In [ ]:
# Adding a map overlay (optional)
import matplotlib.image as mpimg
california_img=mpimg.imread(os.path.join("images", "california.png"))
ax = housing.plot(kind="scatter", x="longitude", y="latitude", figsize=(10,7),
                  s=housing['population']/100, label="Population",
                  c="median_house_value", cmap=plt.get_cmap("jet"),
                  colorbar=False, alpha=0.4)
plt.imshow(california_img, extent=[-124.55, -113.80, 32.45, 42.05], alpha=0.5,
           cmap=plt.get_cmap("jet"))
plt.ylabel("Latitude", fontsize=14)
plt.xlabel("Longitude", fontsize=14)

prices = housing["median_house_value"]
tick_values = np.linspace(prices.min(), prices.max(), 11)
cbar = plt.colorbar(ticks=tick_values/prices.max())
cbar.ax.set_yticklabels(["$%dk"%(round(v/1000)) for v in tick_values], fontsize=14)
cbar.set_label('Median House Value', fontsize=16)

plt.legend(fontsize=16)
save_fig("california_housing_prices_plot")
plt.show()

This image tells you that the **housing prices are very much related to the location** (e.g., close to the ocean) **and to the population density**, as you probably knew already. 

## Looking for Correlations

In [ ]:
import seaborn as sns

corr_matrix = housing.drop("ocean_proximity", axis=1).corr()

# Correlation heatmap
sns.heatmap(corr_matrix, annot=True, fmt='.2f');

Now let’s focus on how much each attribute correlates with the **median house value**.

In [ ]:
corr_matrix["median_house_value"].sort_values(ascending=False)

The most promising attribute to predict the median house value is the **median income**, so let’s zoom in on their correlation scatterplot.

In [ ]:
housing.plot(kind="scatter", x="median_income", y="median_house_value",
             alpha=0.1)
plt.axis([0, 16, 0, 550000])
save_fig("income_vs_house_value_scatterplot")

This plot reveals a few things. 

- The correlation is indeed very strong; we can clearly see the upward trend, and the points are not too dispersed. 
- The price cap that we noticed earlier is clearly visible as a horizontal line at \\$500,000. 
But this plot reveals other less obvious straight lines: a horizontal line around \\$450,000, another around $350,000, perhaps one around \\$280,000. 

## Experimenting with Attribute Combinations

One last thing you may want to do before preparing the data for machine learning algorithms is to try out various **attribute combinations**. 

For example:
- The total number of rooms in a district is not very useful if we don’t know how many households there are. What we really want is the **number of rooms per household**.
- The total number of bedrooms by itself is not very useful: we probably want to **compare** it to the number of rooms.
- The **population per household** also seems like an interesting attribute combination to look at.

In [ ]:
housing["rooms_per_household"] = housing["total_rooms"]/housing["households"]
housing["bedrooms_per_room"] = housing["total_bedrooms"]/housing["total_rooms"]
housing["population_per_household"]=housing["population"]/housing["households"]

And now let’s look at the correlation matrix again.

In [ ]:
corr_matrix = housing.drop("ocean_proximity", axis=1).corr()
corr_matrix["median_house_value"].sort_values(ascending=False)

- The new `bedrooms_per_room` attribute is much more correlated with the median house value than the total number of rooms or bedrooms. Apparently houses with a lower bedroom/room ratio tend to be **more expensive** (negative correlation).
- The number of rooms per household is also more informative than the total number of rooms in a district. Obviously the larger the houses, the more expensive they are.

# Preparing the Data for Machine Learning Algorithms

## Data Cleaning

Let’s revert to a clean training set (by copying `train_set` once again). Let’s also **separate the predictors and the labels**, since we don’t necessarily want to apply the same transformations to the predictors and the target values.

In [ ]:
housing = train_set.drop("median_house_value", axis=1) # drop labels for training set
housing_labels = train_set["median_house_value"].copy()

We saw earlier that the `total_bedrooms` attribute has some missing values.

In [ ]:
sample_incomplete_rows = housing[housing.isnull().any(axis=1)].head()
sample_incomplete_rows

Let’s fix this. You have three options:
1. Get rid of the corresponding districts.
2. Get rid of the whole attribute.
3. Set the values to some value (zero, the mean, the median, etc.).

The third option is usually the preferred one. Let's explore this in details.

Scikit-Learn provides a handy class to take care of missing values: `SimpleImputer`. You need to create a `SimpleImputer` instance, specifying that you want to replace each attribute’s missing values with the median of that attribute.

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")

Remove the text attribute because median can only be calculated on **numerical attributes**:

In [ ]:
housing_num = housing.select_dtypes(include=[np.number]) # only numerical attributes

Now you can fit the imputer instance to the training data using the `.fit()` method.

In [ ]:
imputer.fit(housing_num)

The imputer has simply computed the median of each attribute and stored the result in its `statistics_` instance variable. Only the `total_bedrooms` attribute had missing values, but we cannot be sure that there won’t be any missing values in new data after the system goes live, so it is safer to apply the imputer to all the numerical attributes.

In [ ]:
imputer.statistics_

Check that this is the same as manually computing the median of each attribute:

In [ ]:
housing_num.median().values

Now you can use this “trained” imputer to transform the training set by replacing missing values with the learned medians.

In [ ]:
X = imputer.transform(housing_num)

The result is a plain NumPy array containing the transformed features. To put it back into a pandas DataFrame,

In [ ]:
housing_tr = pd.DataFrame(X, columns=housing_num.columns, index=housing.index)

In [ ]:
housing_tr.loc[sample_incomplete_rows.index.values]

## Handling Text and Categorical Attributes

Now let's preprocess the categorical input feature, `ocean_proximity`:

In [ ]:
housing_cat = housing[["ocean_proximity"]]
housing_cat.head(10)

It’s not arbitrary text: there are a limited number of possible values, each of which represents a category. Therefore, `ocean_proximity` is a **categorical attribute**.

Most machine learning algorithms work with numerical data, so we need to convert these categories into a numerical representation. Since there is no natural ordering among the categories (e.g., `INLAND` is not greater or less than `NEAR BAY`), assigning a single number to each category may introduce a misleading relationship between them.

A common solution is to create one binary attribute per category: one attribute equal to 1 when the category is `<1H OCEAN` (and 0 otherwise), another attribute equal to 1 when the category is `INLAND` (and 0 otherwise), and so on. This technique is called **one-hot encoding**.

Scikit-Learn provides the `OneHotEncoder` class to perform this transformation automatically.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_encoder = OneHotEncoder(sparse_output=False)
housing_cat_1hot = cat_encoder.fit_transform(housing_cat)
housing_cat_1hot

We set `sparse_output=False` so that the encoder returns a NumPy array instead of a SciPy sparse matrix. You can view the categories discovered by the encoder using the `categories_` attribute:

In [ ]:
cat_encoder.categories_

## Custom Transformers

Let's create a custom transformer to add the following extra attributes.
- `rooms_per_household`
- `population_per_household`
- `bedrooms_per_room` (optional)

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

# column index
col_names = "total_rooms", "total_bedrooms", "population", "households"
rooms_ix, bedrooms_ix, population_ix, households_ix = [
    housing.columns.get_loc(c) for c in col_names] # get the column indices

class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, add_bedrooms_per_room=True): 
        self.add_bedrooms_per_room = add_bedrooms_per_room
    def fit(self, X, y=None):
        return self  # nothing else to do
    def transform(self, X):
        rooms_per_household = X[:, rooms_ix] / X[:, households_ix] # create rooms_per_household
        population_per_household = X[:, population_ix] / X[:, households_ix] # create population_per_household
        if self.add_bedrooms_per_room:
            bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix]
            return np.c_[X, rooms_per_household, population_per_household,
                         bedrooms_per_room]
        else:
            return np.c_[X, rooms_per_household, population_per_household]

attr_adder = CombinedAttributesAdder(add_bedrooms_per_room=False)
housing_extra_attribs = attr_adder.transform(housing.values)

We package our feature engineering logic into a custom Scikit-Learn transformer called `CombinedAttributesAdder`. This allows feature creation to become part of the preprocessing pipeline, ensuring that the same transformations are automatically applied during training, validation, and testing. Notice that the transformer has a parameter, `add_bedrooms_per_room`, that determines whether the additional feature `bedrooms_per_room` is created.

Also, `housing_extra_attribs` is a NumPy array, we've lost the column names. To recover a `DataFrame`, you could run this:

In [ ]:
housing_extra_attribs = pd.DataFrame(
    housing_extra_attribs,
    columns=list(housing.columns)+["rooms_per_household", "population_per_household"],
    index=housing.index)
housing_extra_attribs.head()

## Feature Scaling

One of the most important transformations you need to apply to your data is feature scaling. Usually, machine learning algorithms don’t perform well when the input numerical attributes have very different scales. This is the case for the housing data: the total number of rooms ranges from about 6 to 39,320, while the median incomes only range from 0 to 15. Note that scaling the target values is generally not required.

There are two common ways to get all attributes to have the same scale: **min-max scaling** and **standardization**.

- Min-max scaling (also known as normalization) is the simplest. Values are shifted and rescaled so that they end up ranging from 0 to 1. Scikit-Learn provides a transformer called `MinMaxScaler` for this. 

- Standardization is different: first it subtracts the mean value (so standardized values always have a zero mean), and then it divides by the standard deviation so that the resulting distribution has unit variance. Unlike min-max scaling, standardization does not bound values to a specific range. However, standardization is much less affected by outliers. For example, suppose a district had a median income equal to 100 (by mistake). Min-max scaling would then crush all the other values from 0–15 down to 0–0.15, whereas standardization would not be much affected. Scikit-Learn provides a transformer called `StandardScaler` for standardization.

Now let's build a pipeline for preprocessing the **numerical attributes**:

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy="median")),
        ('attribs_adder', CombinedAttributesAdder()),
        ('std_scaler', StandardScaler()),
    ])

housing_num_tr = num_pipeline.fit_transform(housing_num)

In [ ]:
housing_num_tr

So far, we have handled the categorical columns and the numerical columns separately. It would be more convenient to have **a single transformer able to handle all columns**, applying the appropriate transformations to each column. Scikit-Learn introduced the `ColumnTransformer` for this purpose. Let’s use it to apply all the transformations to the housing data:

In [ ]:
# Import the `ColumnTransformer` class
from sklearn.compose import ColumnTransformer

# Get the list of numerical column names and the list of categorical column names
num_attribs = list(housing_num)
cat_attribs = ["ocean_proximity"]


full_pipeline = ColumnTransformer([
        # Numerical columns should be transformed using the num_pipeline
        ("num", num_pipeline, num_attribs), 
        # Categorical columns should be transformed using a OneHotEncoder
        ("cat", OneHotEncoder(), cat_attribs),
    ])

housing_prepared = full_pipeline.fit_transform(housing)

In [ ]:
housing_prepared

In [ ]:
housing_prepared.shape

We have a preprocessing pipeline that takes the full housing data and applies the appropriate transformations to each column.

# Select and Train a Model 

## Train a Model

We are now ready to select and train a machine learning model. Let’s first train a linear regression model.

In [ ]:
from sklearn.linear_model import LinearRegression
 
lin_reg = LinearRegression()
lin_reg.fit(housing_prepared, housing_labels)

Done! You now have a working linear regression model. Let’s try it out on a few instances from the training set.

In [ ]:
# let's try the full preprocessing pipeline on the first 5 training data
some_data = housing.iloc[:5] 
some_labels = housing_labels.iloc[:5]
some_data_prepared = full_pipeline.transform(some_data)

print("Predictions:", lin_reg.predict(some_data_prepared))

Compare against the actual values:

In [ ]:
print("Labels:", list(some_labels))

It works, although the predictions are not exactly accurate (e.g., the first prediction is off by close to 40%!). Let’s measure this regression model’s RMSE on the whole training set using Scikit-Learn’s `mean_squared_error()` function.

In [ ]:
from sklearn.metrics import mean_squared_error

housing_predictions = lin_reg.predict(housing_prepared)
lin_rmse = np.sqrt(mean_squared_error(housing_labels, housing_predictions)).item()
lin_rmse

This is better than nothing, but clearly not a great score: most districts’ `median_housing_values` range between \\$120,000 and \\$265,000, so a typical prediction error of \\$68,628 is not very satisfying. This is an example of a model **underfitting** the training data. When this happens it can mean that the features do not provide enough information to make good predictions, or that the model is not powerful enough. 

The main ways to fix underfitting are to select a **more powerful model**, to feed the training algorithm with better features, or to reduce the constraints on the model. This model is not regularized, which rules out the last option. You could try to add more features (e.g., the log of the population), but first let’s try a more complex model to see how it does.

Let’s train a `DecisionTreeRegressor`. This is a powerful model, capable of finding complex nonlinear relationships in the data.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = DecisionTreeRegressor(random_state=100)
tree_reg.fit(housing_prepared, housing_labels)

Now that the model is trained, let’s evaluate it on the training set.

In [ ]:
housing_predictions = tree_reg.predict(housing_prepared)
tree_mse = mean_squared_error(housing_labels, housing_predictions)
tree_rmse = np.sqrt(tree_mse)
tree_rmse

Wait, what!? No error at all? Could this model really be absolutely perfect? Of course, it is much more likely that the model has badly **overfit** the data. How can we be sure?

As we saw earlier, you don’t want to touch the test set until you are ready to launch a model you are confident about, so you need to use part of the training set for training and part of it for model validation.

## Better Evaluation Using Cross-Validation

One way to evaluate the decision tree model would be to use the `train_test_split()` function to split the training set into a smaller training set and a validation set, then train your models against the smaller training set and evaluate them against the validation set. One can do this using Scikit-Learn’s $K$-fold cross-validation feature. The following code randomly splits the training set into 4 distinct subsets called folds, then it trains and evaluates the decision tree model 4 times, picking a different fold for evaluation every time and training on the other 3 folds. The result is an array containing the 4 evaluation scores. Note that **models with lower score are better**.

In [ ]:
from sklearn.model_selection import cross_val_score

# CV score of decision tree regression model
scores = cross_val_score(tree_reg, housing_prepared, housing_labels,
                         scoring="neg_mean_squared_error", cv=4)
tree_rmse_scores = np.sqrt(-scores)

In [ ]:
def display_scores(scores):
    print("Scores:", scores)
    print("Mean:", scores.mean())
    print("Standard deviation:", scores.std())

display_scores(tree_rmse_scores)

In [ ]:
# CV score of linear regression model
lin_scores = cross_val_score(lin_reg, housing_prepared, housing_labels,
                             scoring="neg_mean_squared_error", cv=4)
lin_rmse_scores = np.sqrt(-lin_scores)
display_scores(lin_rmse_scores)

Now the Decision Tree doesn’t look as good as it did earlier. In fact, it seems to perform worse than the linear regression model. Notice that cross-validation allows you to get not only an estimate of the performance of your model, but also a measure of how precise this estimate is (i.e., its standard deviation). 

Let’s try one more model, the `RandomForestRegressor`. Random Forests work by training many Decision Trees on random subsets of the features, then averaging out their predictions. Building a model on top of many
other models is called Ensemble Learning, and it is often a great way to push machine learning algorithms even further.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest_reg = RandomForestRegressor(n_estimators=20, random_state=100)
forest_reg.fit(housing_prepared, housing_labels)

In [ ]:
housing_predictions = forest_reg.predict(housing_prepared)
forest_mse = mean_squared_error(housing_labels, housing_predictions)
forest_rmse = np.sqrt(forest_mse)
forest_rmse

In [ ]:
forest_scores = cross_val_score(forest_reg, housing_prepared, housing_labels,
                                scoring="neg_mean_squared_error", cv=4)
forest_rmse_scores = np.sqrt(-forest_scores)
display_scores(forest_rmse_scores)

Random forests look very promising. However, note that the score on the training set is still much lower than on the validation sets, meaning that the model is still overfitting the training set. Possible solutions for overfitting are to simplify the model, constrain it (i.e., regularize it), or get a lot more training data.

However, you should try out many other models from various categories of machine learning algorithms (e.g.,
Support Vector Machines), without spending too much time tweaking the hyperparameters. The goal is to shortlist a few (two to five) promising models.

## Fine-Tuning Model

Assume that we now have a shortlist of promising models. We need to fine-tune them. We get Scikit-Learn’s `GridSearchCV` to automate it. All you need to do is tell it which hyperparameters you want it to experiment with and what values to try out, and it will use cross-validation to evaluate all the possible combinations of hyperparameter values. For example, the following code searches for the best combination of hyperparameter values for the `RandomForestRegressor`.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = [
    # try 12 (3×4) combinations of hyperparameters
    {'n_estimators': [3, 10, 30], 'max_features': [2, 4, 6, 8]},
    # then try 6 (2×3) combinations with bootstrap set as False
    {'bootstrap': [False], 'n_estimators': [3, 10], 'max_features': [2, 3, 4]},
  ]

forest_reg = RandomForestRegressor(random_state=100)
# train across 5 folds, that's a total of (12+6)*5=90 rounds of training 
grid_search = GridSearchCV(forest_reg, param_grid, cv=4,
                           scoring='neg_mean_squared_error',
                           return_train_score=True)
grid_search.fit(housing_prepared, housing_labels)

The best hyperparameter combination found:

In [ ]:
grid_search.best_params_

In [ ]:
grid_search.best_estimator_

Let's look at the score of each hyperparameter combination tested during the grid search:

In [ ]:
cvres = grid_search.cv_results_
for mean_score, params in zip(cvres["mean_test_score"], cvres["params"]):
    print(np.sqrt(-mean_score), params)

## Evaluate Your System on the Test Set

After tweaking your models for a while, you eventually have a system that performs sufficiently well. Now is the time to evaluate the final model on the test set.

In [ ]:
final_model = grid_search.best_estimator_

X_test = test_set.drop("median_house_value", axis=1)
y_test = test_set["median_house_value"].copy()

X_test_prepared = full_pipeline.transform(X_test)
final_predictions = final_model.predict(X_test_prepared)

final_mse = mean_squared_error(y_test, final_predictions)
final_rmse = np.sqrt(final_mse)

In [ ]:
final_rmse

We can compute a 95% confidence interval for the test RMSE:

In [ ]:
from scipy import stats

confidence = 0.95
squared_errors = (final_predictions - y_test) ** 2
np.sqrt(stats.t.interval(confidence, len(squared_errors) - 1,
                         loc=squared_errors.mean(),
                         scale=stats.sem(squared_errors)))

Now comes the project prelaunch phase: we need to present the solution by
- highlighting what you have learned
- what worked and what did not
- what assumptions were made, and
- what your system’s limitations are

Document everything, and create nice presentations with clear visualizations and easy-to-remember statements (e.g., “the median income is the number one predictor of housing prices”). In this California housing example, the final performance of the system is not better than the experts’ price estimates, which were often off by about 20%, but it may still be a good idea to launch it, especially if this frees up some time for the experts so they can work on more interesting and productive tasks.